In [12]:
### SQL DATABASES
import sqlite3
import os 

os.makedirs("data/databases", exist_ok=True)

In [13]:
conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

print("Database created successfully")

Database created successfully


In [14]:
cursor.execute('''CREATE TABLE IF NOT EXISTS employees
             (id INTEGER PRIMARY KEY, name TEXT,role TEXT,department TEXT, salary REAL)''')

In [15]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects
               (id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget REAL, lead_id INTERGER)
''')

In [16]:
cursor.executemany("""
INSERT INTO employees (id, name, role, department, salary)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, "Amit Sharma", "Data Scientist", "AI", 120000),
    (2, "Priya Verma", "Backend Developer", "Engineering", 95000),
    (3, "Rahul Mehta", "Project Manager", "Operations", 110000),
    (4, "Sneha Kapoor", "Frontend Developer", "Engineering", 90000),
    (5, "Arjun Singh", "ML Engineer", "AI", 115000)
])

conn.commit()

In [ ]:
cursor.executemany("""
INSERT INTO projects (id, name, status, budget, lead_id)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, "Churn Prediction System", "Ongoing", 500000, 1),
    (2, "E-commerce Backend", "Completed", 300000, 2),
    (3, "Customer Analytics Dashboard", "Ongoing", 200000, 3),
    (4, "Website Redesign", "Completed", 150000, 4),
    (5, "Recommendation Engine", "Ongoing", 400000, 5)
])

conn.commit()
conn.close()

In [19]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

/Users/mayanksharma/Desktop/Projects/Rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
db= SQLDatabase.from_uri("sqlite:///data/databases/company.db")

print(f'Tables: {db.get_usable_table_names()}')
print(f'\nTables DDL:')
print(db.get_table_info())

Tables: ['employees', 'projects']

Tables DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Amit Sharma	Data Scientist	AI	120000.0
2	Priya Verma	Backend Developer	Engineering	95000.0
3	Rahul Mehta	Project Manager	Operations	110000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	Churn Prediction System	Ongoing	500000.0	1
2	E-commerce Backend	Completed	300000.0	2
3	Customer Analytics Dashboard	Ongoing	200000.0	3
*/


In [31]:
from typing import List
import sqlite3
from langchain_core.documents import Document

def sql_to_documents(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents = []

    # Get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        # Get columns
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        # Get data
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()

        # Build content
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        table_content += "Sample Data:\n"
        for row in rows[:5]:
            record = dict(zip(column_names, row))
            table_content += f"{record}\n"

        doc = Document(
            page_content=table_content,
            metadata={
                "source": db_path,
                "table_name": table_name,
                "num_records": len(rows),
                "data_type": "sql"
            }
        )

        documents.append(doc)

    # ✅ Relationship doc (ONLY ONCE)
    try:
        cursor.execute("""
            SELECT e.name, e.role, p.name as project_name, p.status
            FROM employees e
            JOIN projects p ON e.id = p.lead_id
        """)

        relationships = cursor.fetchall()

        rel_content = "Employee-Project Relationship:\n\n"
        for rel in relationships:
            rel_content += f"Manager: {rel[0]}, Role: {rel[1]}, Project: {rel[2]}, Status: {rel[3]}\n"

        rel_doc = Document(
            page_content=rel_content,
            metadata={
                "source": db_path,
                "table_name": "employee_project_relationship",
                "data_type": "sql_relationship"
            }
        )

        documents.append(rel_doc)

    except Exception as e:
        print("Relationship query skipped:", e)

    conn.close()
    return documents

In [32]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_records': 5, 'data_type': 'sql'}, page_content="Table: employees\nColumns: id, name, role, department, salary\nTotal Records: 5\n\nSample Data:\n{'id': 1, 'name': 'Amit Sharma', 'role': 'Data Scientist', 'department': 'AI', 'salary': 120000.0}\n{'id': 2, 'name': 'Priya Verma', 'role': 'Backend Developer', 'department': 'Engineering', 'salary': 95000.0}\n{'id': 3, 'name': 'Rahul Mehta', 'role': 'Project Manager', 'department': 'Operations', 'salary': 110000.0}\n{'id': 4, 'name': 'Sneha Kapoor', 'role': 'Frontend Developer', 'department': 'Engineering', 'salary': 90000.0}\n{'id': 5, 'name': 'Arjun Singh', 'role': 'ML Engineer', 'department': 'AI', 'salary': 115000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'projects', 'num_records': 5, 'data_type': 'sql'}, page_content="Table: projects\nColumns: id, name, status, budget, lead_id\nTotal Records: 5\n\nSample Data